# Asset List & Rental Data Migration (AMS to SMART)
### Staging ETL Pipeline: Cleaning, Code Deduplication, Rental Sorting, & Subcategory Partitioning

## 1. Overview & Business Context
This notebook executes the foundational ETL data preparation pipeline to migrate asset inventory and rental records from legacy AMS (`RE_ASSETS.dbo.tb_assets_list` and `RE_ASSETS.dbo.tb_assets_rent`) into the Smart Enterprise Asset Management system.

### Key Pipeline Objectives:
1. **Step 1: Overall Cleaning**:
   - Filter out **37 dummy / testing items** (`asfg`, `asdf`, `dev-test`, `test dev`, etc.) from `tb_assets_list`.
   - Preserve all **6,962 legitimate asset records**.
2. **Legacy Code Deduplication**:
   - Resolve **27 code collisions** in AMS where multiple records share the same code.
   - For deleted items (`deletion == 1`): generate a new unique code and record the original in `duplicated_number`.
   - For single active items (`deletion == 0`): retain the original code.
   - For multiple active items (`deletion == 0`): primary active retains original code; colliding active items receive newly serialized codes with the original recorded in `duplicated_number`.
   - Serialization: **CFS** sequence begins at `04884`, **ICT** sequence begins at `2504`.
   - Result: **0 duplicate codes** remaining across all 6,962 assets.
3. **Step 2: Rent Table Sorting**:
   - Sort `tb_assets_rent` by effective event date (`end_date` if present, else fallback to `start_date`).
   - Secondary deterministic ordering: `start_date ASC, id ASC`.
4. **Step 3: Subcategory Partitioning & Refined Anomaly Resolution**:
   - Map each asset to its SMART subcategory using `data/smart/staging/subcategory/subcategory_ams_mapping.csv`.
   - Resolve **legacy anomaly rows and item-specific overrides**:
     - Portable ACs (`AC Fortable`, `Ac Portable`) -> `PDGR-ACP`.
     - Cassette ACs (`AC Casete`, `Ac Casete`) including code `02666-ACC-AC-CFS-PTRE-12` -> `PDGR-ACN`.
     - `Blower Industri` -> new subcategory `ELEK-EXH` (Exhaust).
     - Other `260000` items -> `ELEK-CS`, `ELEK-HD`, `ELEK-SD`, `ELEK-SPR`, `MON-LCD`, `ELEK-TV`.
     - Category anomalies: `180001` -> `PDGR-ACSP`, `120000` -> `COMP-NB`, `2400010` -> `ELEK-CCT`, etc.
   - Partition cleaned assets into **111 individual CSV files per subcategory** (`data/smart/staging/assets/<subcategory_code>.csv`).
   - Enforce rigorous integrity assertions: **0 duplicate IDs** across and within files, set partition invariant.


In [1]:
import pandas as pd
import numpy as np
import json
from collections import Counter, defaultdict
from pathlib import Path

# ---------------------------------------------------------
# Dynamic Path Configurations
# ---------------------------------------------------------
def get_data_dir():
    candidates = [
        Path("data"),
        Path("../data"),
        Path("/home/jovyan/data"),
        Path("/home/ran/Devs/ams-smart-migration/data")
    ]
    for p in candidates:
        if p.exists() and (p / "ams").exists():
            return p.resolve()
    return Path("data").resolve()

DATA_DIR = get_data_dir()
AMS_DIR = DATA_DIR / "ams"
STAGING_DIR = DATA_DIR / "smart" / "staging"
ASSETS_STAGING_DIR = STAGING_DIR / "assets"
RENT_STAGING_DIR = STAGING_DIR / "rent"
SUBCATEGORY_DIR = STAGING_DIR / "subcategory"

ASSETS_STAGING_DIR.mkdir(parents=True, exist_ok=True)
RENT_STAGING_DIR.mkdir(parents=True, exist_ok=True)

print(f"Resolved Data Directory: {DATA_DIR}")


Resolved Data Directory: /home/jovyan/data


## 2. Load Raw Datasets
We load the raw legacy assets table `tb_assets_list`, borrowing table `tb_assets_rent`, and subcategories master.


In [2]:
raw_asset_file = sorted(AMS_DIR.glob("tb_assets_list_*.csv"))[-1]
raw_rent_file = sorted(AMS_DIR.glob("tb_assets_rent_*.csv"))[-1]
mapping_file = SUBCATEGORY_DIR / "subcategory_ams_mapping.csv"

print(f"Loading assets from: {raw_asset_file.name}")
print(f"Loading rentals from: {raw_rent_file.name}")

df_assets_raw = pd.read_csv(raw_asset_file, dtype=str, keep_default_na=False)
df_rent_raw = pd.read_csv(raw_rent_file, dtype=str, keep_default_na=False)
df_mapping = pd.read_csv(mapping_file, dtype=str, keep_default_na=False)

print(f"Total raw assets: {len(df_assets_raw):,}")
print(f"Total raw rentals: {len(df_rent_raw):,}")
print(f"Total subcategory mappings: {len(df_mapping):,}")


Loading assets from: tb_assets_list_202610021221.csv
Loading rentals from: tb_assets_rent_202610021222.csv
Total raw assets: 6,999
Total raw rentals: 2,348
Total subcategory mappings: 113


## 3. Step 1: Overall Cleaning (Dummy & Test Data Removal)
Filter out test and dummy asset records created during testing phases.


In [3]:
DUMMY_NAMES = {
    'asfg', 'asdf', 'ASDF', 'Asdfertg', 'dfjdjg', 'test', 'INI BARANG TEST',
    'da', 'ssss', '!!!zzzz', 'Test', 'asdad', 'afsd', 'asdffadg', 'aserfd',
    'dev-test', 'test dev', 'TestDev001', 'test dev002', 'test dev 002',
    'test dev 003', 'test dev', 'test dev 00123', 'TestDev004', 'sdf', 'Test Dev',
    'TestDev006', 'TestDev007'
}
DUMMY_LOWER = {n.strip().lower() for n in DUMMY_NAMES}

is_dummy = df_assets_raw["name"].str.strip().str.lower().isin(DUMMY_LOWER)
df_dummies = df_assets_raw[is_dummy].copy()
df_assets_cleaned = df_assets_raw[~is_dummy].copy()

print(f"Dummy records removed: {len(df_dummies)}")
print(f"Cleaned asset records: {len(df_assets_cleaned):,}")

assert len(df_dummies) == 37, f"Expected 37 dummies, got {len(df_dummies)}"
assert len(df_assets_cleaned) == 6962, f"Expected 6,962 cleaned, got {len(df_assets_cleaned)}"


Dummy records removed: 37
Cleaned asset records: 6,962


## 4. Legacy Asset Code Deduplication
In legacy AMS, 27 code collisions exist where multiple asset rows share the exact same asset `code`.
We resolve collisions deterministically:
- **Deleted items (`deletion == 1`)**: Re-serialize with a new unique code, storing the original in `duplicated_number`.
- **Sole active item (`deletion == 0`)**: Retain original code (`duplicated_number = ""`).
- **Multiple active items (`deletion == 0`)**: Primary (earliest) retains original code, subsequent colliding items re-serialized with original stored in `duplicated_number`.
- **Serial Sequences**: CFS starts at `04884`, ICT starts at `2504`.


In [4]:
code_counts = df_assets_cleaned["code"].value_counts()
collision_codes = code_counts[code_counts > 1].index.tolist()
print(f"Total colliding legacy codes: {len(collision_codes)}")

cfs_cur = 4884
ict_cur = 2504

updates = {}
audit_dedup = []

for code in sorted(collision_codes):
    matching = df_assets_cleaned[df_assets_cleaned["code"] == code].copy()
    matching["del_int"] = matching["deletion"].astype(int)
    matching["id_int"] = matching["id"].astype(int)
    matching = matching.sort_values(by=["del_int", "id_int"])
    
    active = matching[matching["deletion"] == "0"].to_dict("records")
    deleted = matching[matching["deletion"] == "1"].to_dict("records")
    
    if len(active) > 0:
        primary = active[0]
        audit_dedup.append({
            "id": primary["id"],
            "original_code": primary["code"],
            "final_code": primary["code"],
            "duplicated_number": "",
            "deletion": primary["deletion"],
            "status": "retained_original_primary_active"
        })
        for act in active[1:]:
            organizer = act.get("organizer", "")
            sep = "-" if "-" in act["code"] else " "
            rest = act["code"].split(sep, 1)[1] if sep in act["code"] else act["code"]
            if "CFS" in act["code"] or organizer == "CFS":
                new_serial = f"{cfs_cur:05d}"
                cfs_cur += 1
            else:
                new_serial = f"{ict_cur}"
                ict_cur += 1
            new_code = f"{new_serial}-{rest.lstrip('-')}"
            updates[act["id"]] = (new_code, act["code"])
            audit_dedup.append({
                "id": act["id"],
                "original_code": act["code"],
                "final_code": new_code,
                "duplicated_number": act["code"],
                "deletion": act["deletion"],
                "status": "renumbered_duplicate_active"
            })
            
    for d in deleted:
        organizer = d.get("organizer", "")
        sep = "-" if "-" in d["code"] else " "
        rest = d["code"].split(sep, 1)[1] if sep in d["code"] else d["code"]
        if "CFS" in d["code"] or organizer == "CFS":
            new_serial = f"{cfs_cur:05d}"
            cfs_cur += 1
        else:
            new_serial = f"{ict_cur}"
            ict_cur += 1
        new_code = f"{new_serial}-{rest.lstrip('-')}"
        updates[d["id"]] = (new_code, d["code"])
        audit_dedup.append({
            "id": d["id"],
            "original_code": d["code"],
            "final_code": new_code,
            "duplicated_number": d["code"],
            "deletion": d["deletion"],
            "status": "renumbered_deleted"
        })

# Apply updates
df_assets_cleaned["duplicated_number"] = ""
for row_id, (new_c, dup_c) in updates.items():
    idx = df_assets_cleaned[df_assets_cleaned["id"] == row_id].index
    df_assets_cleaned.loc[idx, "code"] = new_c
    df_assets_cleaned.loc[idx, "duplicated_number"] = dup_c

print(f"Colliding records renumbered: {len(updates)}")
print(f"Next available CFS serial: {cfs_cur:05d}")
print(f"Next available ICT serial: {ict_cur}")

# Verify zero duplicate codes remain
final_code_counts = df_assets_cleaned["code"].value_counts()
remaining_collisions = final_code_counts[final_code_counts > 1]
assert len(remaining_collisions) == 0, f"Duplicates still present: {remaining_collisions}"
print("Assertion Passed: 0 duplicate codes remaining across all 6,962 rows!")


Total colliding legacy codes: 27
Colliding records renumbered: 30
Next available CFS serial: 04910
Next available ICT serial: 2508
Assertion Passed: 0 duplicate codes remaining across all 6,962 rows!


## 5. Step 2: Sorting tb_assets_rent
Sort rental records by effective date (`end_date` if exists, else fallback to `start_date`) with deterministic tie-breaker (`start_date ASC, id ASC`).


In [5]:
df_rent = df_rent_raw.copy()

def get_effective_date(row):
    end_d = row["end_date"].strip()
    start_d = row["start_date"].strip()
    if end_d and end_d != "NULL":
        return end_d
    if start_d and start_d != "NULL":
        return start_d
    return "9999-12-31"

def get_start_date(row):
    start_d = row["start_date"].strip()
    return start_d if start_d and start_d != "NULL" else "9999-12-31"

df_rent["effective_date"] = df_rent.apply(get_effective_date, axis=1)
df_rent["start_date_sort"] = df_rent.apply(get_start_date, axis=1)
df_rent["id_int"] = df_rent["id"].astype(int)

df_rent_sorted = df_rent.sort_values(by=["effective_date", "start_date_sort", "id_int"]).copy()
df_rent_sorted = df_rent_sorted.drop(columns=["effective_date", "start_date_sort", "id_int"])

sorted_rent_path = RENT_STAGING_DIR / "tb_assets_rent_sorted.csv"
df_rent_sorted.to_csv(sorted_rent_path, index=False)
print(f"Saved sorted rental records ({len(df_rent_sorted):,} rows) to: {sorted_rent_path.name}")


Saved sorted rental records (2,348 rows) to: tb_assets_rent_sorted.csv


## 6. Step 3: Subcategory Partitioning & Refined Anomaly Resolution
Map AMS category codes to SMART subcategories, applying item-specific overrides:
- Portable ACs (`1135`, `7957`) -> `PDGR-ACP`
- Cassette ACs (`1431`, `1432`, `7984`, `7985`, `7986`) -> `PDGR-ACN` (including code `02666-ACC-AC-CFS-PTRE-12`)
- Industrial Blower (`8576`) -> new subcategory `ELEK-EXH` (Exhaust)
- Other `260000` items -> `ELEK-CS`, `ELEK-HD`, `ELEK-SD`, `ELEK-SPR`, `MON-LCD`, `ELEK-TV`
- General category anomalies: `180001` -> `PDGR-ACSP`, `120000` -> `COMP-NB`, `2400010` -> `ELEK-CCT`, etc.


In [6]:
code_to_sub = dict(zip(df_mapping["ams_code"], df_mapping["subcategory_code"]))

# Case 1: Realized data entry blunders (deletion == 1) -> permanently_deleted_assets.csv
PERMANENTLY_DELETED_ANOMALIES = {
    # 8 AC Splits under ELEK-MK (240005)
    '752': 'ac_split_in_elek_mk_deleted',
    '2089': 'ac_split_in_elek_mk_deleted',
    '2090': 'ac_split_in_elek_mk_deleted',
    '7946': 'ac_split_in_elek_mk_deleted',
    '7947': 'ac_split_in_elek_mk_deleted',
    '7948': 'ac_split_in_elek_mk_deleted',
    '7960': 'ac_split_in_elek_mk_deleted',
    '8947': 'ac_split_in_elek_mk_deleted',
    # 5 Floor Carpets under PERP-ACCS (140016)
    '8575': 'carpet_in_perp_accs_deleted',
    '8686': 'carpet_in_perp_accs_deleted',
    '8949': 'carpet_in_perp_accs_deleted',
    '8950': 'carpet_in_perp_accs_deleted',
    '9462': 'carpet_in_perp_accs_deleted',
    # 1 Glass Rack under PERP-ACCS (140016)
    '9368': 'dish_rack_in_perp_accs_deleted',
    # 1 Dispenser under PDGR-ACS (260003)
    '8728': 'dispenser_in_pdgr_acs_deleted',
    # 1 Office Desk under FUR-LC (100003)
    '861': 'meja_in_fur_lc_deleted',
    # 1 Ceiling Speaker under ELEK-TV (240002)
    '8214': 'ceiling_speaker_in_elek_tv_deleted',
    # 1 Casing under TLKM-SP
    '10732': 'casing_in_tlkm_sp_deleted',
}

# Case 2: Active physical assets (deletion == 0) -> Moved to correct subcategory, code preserved
ITEM_SPECIFIC_OVERRIDES = {
    # Previously resolved overrides
    '7984': ('PDGR-ACN', 'bonus_anomaly_02666_ac_casete'),
    '1135': ('PDGR-ACP', 'portable_ac_override'),
    '7957': ('PDGR-ACP', 'portable_ac_override'),
    '1431': ('PDGR-ACN', 'casete_ac_override'),
    '1432': ('PDGR-ACN', 'casete_ac_override'),
    '7985': ('PDGR-ACN', 'casete_ac_override'),
    '7986': ('PDGR-ACN', 'casete_ac_override'),
    
    # 8 Ceiling Speakers in ELEK-TV -> ELEK-CS
    '8207': ('ELEK-CS', 'ceiling_speaker_misassigned_in_tv'),
    '8215': ('ELEK-CS', 'ceiling_speaker_misassigned_in_tv'),
    '8221': ('ELEK-CS', 'ceiling_speaker_misassigned_in_tv'),
    '8222': ('ELEK-CS', 'ceiling_speaker_misassigned_in_tv'),
    '8225': ('ELEK-CS', 'ceiling_speaker_misassigned_in_tv'),
    '8227': ('ELEK-CS', 'ceiling_speaker_misassigned_in_tv'),
    '8232': ('ELEK-CS', 'ceiling_speaker_misassigned_in_tv'),
    '8235': ('ELEK-CS', 'ceiling_speaker_misassigned_in_tv'),
    
    # 10 Topcon Surveying & Geodetic Tools in MES-RP -> ELEK-TG
    '8124': ('ELEK-TG', 'topcon_total_station_surveying'),
    '8125': ('ELEK-TG', 'topcon_tripod_surveying'),
    '8126': ('ELEK-TG', 'topcon_tripod_surveying'),
    '8127': ('ELEK-TG', 'topcon_prisma_polygon_surveying'),
    '8128': ('ELEK-TG', 'topcon_auto_level_surveying'),
    '8129': ('ELEK-TG', 'topcon_battery_surveying'),
    '8130': ('ELEK-TG', 'topcon_usb_surveying'),
    '8131': ('ELEK-TG', 'topcon_strap_surveying'),
    '8132': ('ELEK-TG', 'topcon_software_surveying'),
    '8133': ('ELEK-TG', 'topcon_reflector_surveying'),
    
    # Scaffolding & Paging in MES-RP
    '8138': ('ACC-STB', 'scaffolding_in_mes_rp'),
    '8141': ('ELEK-SPE', 'paging_samsung_in_mes_rp'),
    
    # TVs, CCTV DVRs, and Speakers in PERP-MMD
    '2661': ('ELEK-TV', 'plasma_tv_in_perp_mmd'),
    '6563': ('ELEK-TV', 'smart_tv_in_perp_mmd'),
    '7941': ('ELEK-TV', 'smart_tv_in_perp_mmd'),
    '3640': ('ELEK-CCT', 'cctv_dvr_in_perp_mmd'),
    '6564': ('ELEK-CCT', 'cctv_dvr_in_perp_mmd'),
    '6565': ('ELEK-CCT', 'cctv_dvr_in_perp_mmd'),
    '2983': ('ELEK-SPE', 'speaker_aktif_in_perp_mmd'),
    '3757': ('ELEK-SPE', 'pa_portable_in_perp_mmd'),
    
    # TV in MON-LCD
    '8118': ('ELEK-TV', 'smart_tv_50_in_mon_lcd'),
    
    # Xerox A0 Plotter in PRNT-LJT
    '2966': ('PRNT-PLT', 'xerox_a0_plotter_in_prnt_ljt'),
    
    # Heat Detector in ELEK-SD
    '12004': ('ELEK-HD', 'heat_detector_in_elek_sd'),
    
    # Drawers in FUR-KK
    '1375': ('FUR-LC', 'laci_kerja_in_fur_kk'),
    '1387': ('FUR-LC', 'laci_kerja_in_fur_kk'),
    
    # First Aid Box in PERP-ACCS
    '8633': ('ACC-P3K', 'kotak_p3k_in_perp_accs'),
}

ANOMALY_CATEGORY_MAP = {
    '180001': 'PDGR-ACSP',
    '120000': 'COMP-NB',
    '2400010': 'ELEK-CCT',
    '1100013': 'ELEK-MIC',
    '2400004': 'ACC-GE',
    '180006': 'ELEK-KUL',
    '1100011': 'ACC-DP',
    '2400013': 'ELEK-TV',
}

ANOMALY_260000_MAP = {
    '8209': 'ELEK-CS',
    '8354': 'ELEK-HD',
    '8423': 'ELEK-SD',
    '8477': 'ELEK-SPR',
    '8507': 'ELEK-SPR',
    '8576': 'ELEK-EXH',  # New Exhaust subcategory!
    '8698': 'MON-LCD',
    '9394': 'ELEK-TV',
    '9456': 'ELEK-HD',
}

def resolve_subcategory(row):
    cat = row["category"].strip()
    row_id = str(row["id"]).strip()
    if row_id in PERMANENTLY_DELETED_ANOMALIES:
        return "PERMANENTLY_DELETED", PERMANENTLY_DELETED_ANOMALIES[row_id]
    if row_id in ITEM_SPECIFIC_OVERRIDES:
        sub_code, reason = ITEM_SPECIFIC_OVERRIDES[row_id]
        return sub_code, reason
    if cat == "260000" and row_id in ANOMALY_260000_MAP:
        return ANOMALY_260000_MAP[row_id], "anomaly_260000"
    if cat in ANOMALY_CATEGORY_MAP:
        return ANOMALY_CATEGORY_MAP[cat], f"anomaly_cat_{cat}"
    if cat in code_to_sub:
        return code_to_sub[cat], "direct"
    raise ValueError(f"Unmapped row {row_id} with category {cat}")

sub_res = [resolve_subcategory(r) for _, r in df_assets_cleaned.iterrows()]
df_assets_cleaned["target_subcategory"] = [s[0] for s in sub_res]
df_assets_cleaned["mapping_type"] = [s[1] for s in sub_res]

# Anomaly audit
df_anomaly_audit = df_assets_cleaned[df_assets_cleaned["mapping_type"] != "direct"][
    ["id", "code", "name", "category", "target_subcategory", "mapping_type"]
].rename(columns={"target_subcategory": "mapped_subcategory"})

anomaly_audit_path = ASSETS_STAGING_DIR / "unmapped_category_audit.csv"
df_anomaly_audit.to_csv(anomaly_audit_path, index=False)
print(f"Resolved {len(df_anomaly_audit)} anomaly & override rows. Audit saved to: {anomaly_audit_path.name}")


Resolved 148 anomaly & override rows. Audit saved to: unmapped_category_audit.csv


## 7. Write Staging Partitions & Validation
Export global cleaned dataset and per-subcategory CSV files, enforcing integrity assertions.


In [7]:
# 1. Global cleaned dataset
global_cleaned_path = ASSETS_STAGING_DIR / "tb_assets_list_cleaned.csv"
export_cols = [c for c in df_assets_cleaned.columns if c not in ["target_subcategory", "mapping_type"]]
df_assets_cleaned[export_cols].to_csv(global_cleaned_path, index=False)
print(f"Saved global cleaned assets to: {global_cleaned_path.name}")

# 2. Duplicate codes audit
dedup_audit_path = ASSETS_STAGING_DIR / "duplicated_codes_audit.csv"
pd.DataFrame(audit_dedup).to_csv(dedup_audit_path, index=False)
print(f"Saved duplicate codes audit to: {dedup_audit_path.name}")

# 3. Export permanently deleted assets (Case 1)
is_perm_del = df_assets_cleaned["target_subcategory"] == "PERMANENTLY_DELETED"
df_permanently_deleted = df_assets_cleaned[is_perm_del].copy()
perm_del_path = ASSETS_STAGING_DIR / "permanently_deleted_assets.csv"
df_permanently_deleted[export_cols].to_csv(perm_del_path, index=False)
print(f"Saved {len(df_permanently_deleted)} permanently deleted assets to: {perm_del_path.name}")

# 4. Export per-subcategory CSV files (111 subcategories from active split)
df_subcategories_split = df_assets_cleaned[~is_perm_del].copy()
sub_master_file = SUBCATEGORY_DIR / "subcategories.csv"
df_sub_master = pd.read_csv(sub_master_file, dtype=str, keep_default_na=False)
all_master_codes = sorted(df_sub_master["code"].unique())

split_groups = {sub_code: grp for sub_code, grp in df_subcategories_split.groupby("target_subcategory")}
manifest = {}
seen_ids = set()
total_partition_rows = 0

for sub_code in all_master_codes:
    sub_file_path = ASSETS_STAGING_DIR / f"{sub_code}.csv"
    if sub_code in split_groups:
        group = split_groups[sub_code]
        group[export_cols].to_csv(sub_file_path, index=False)
        
        item_ids = group["id"].tolist()
        assert len(item_ids) == len(set(item_ids)), f"Duplicate IDs inside {sub_code}.csv"
        
        overlap = seen_ids.intersection(set(item_ids))
        assert len(overlap) == 0, f"Cross-file duplicate IDs in {sub_code}.csv: {overlap}"
        seen_ids.update(item_ids)
        
        row_count = len(group)
        total_partition_rows += row_count
    else:
        # Empty partition (e.g. MES-RP whose misentries were all relocated)
        empty_df = pd.DataFrame(columns=export_cols)
        empty_df.to_csv(sub_file_path, index=False)
        row_count = 0
        
    manifest[sub_code] = {
        "file": f"{sub_code}.csv",
        "row_count": row_count
    }

# Also verify that permanently deleted IDs do not overlap with partitioned IDs
perm_del_ids = set(df_permanently_deleted["id"].tolist())
perm_overlap = seen_ids.intersection(perm_del_ids)
assert len(perm_overlap) == 0, f"Permanently deleted IDs overlap with active partitions: {perm_overlap}"

# 5. Manifest
manifest_path = ASSETS_STAGING_DIR / "_split_manifest.json"
with open(manifest_path, mode="w", encoding="utf-8") as f:
    json.dump({
        "total_assets_cleaned": len(df_assets_cleaned),
        "total_partitioned_assets": total_partition_rows,
        "total_permanently_deleted_assets": len(df_permanently_deleted),
        "total_subcategories": len(manifest),
        "subcategories": manifest
    }, f, indent=2)
print(f"Saved partition manifest to: {manifest_path.name}")

# Final verification assertions
assert len(df_assets_cleaned) == 6962, f"Expected 6,962 cleaned assets, got {len(df_assets_cleaned)}"
assert len(df_permanently_deleted) == 18, f"Expected 18 permanently deleted assets, got {len(df_permanently_deleted)}"
assert total_partition_rows == 6944, f"Expected 6,944 partitioned rows, got {total_partition_rows}"
assert total_partition_rows + len(df_permanently_deleted) == 6962, "Integrity check failed: partitioned + deleted != total"
assert len(seen_ids) == 6944, f"Expected 6,944 unique partitioned IDs, got {len(seen_ids)}"
assert len(manifest) == 111, f"Expected 111 subcategories, got {len(manifest)}"

# Verify specific overrides
assert len(df_subcategories_split[df_subcategories_split["target_subcategory"] == "ELEK-EXH"]) == 1, "ELEK-EXH should have 1 item (Blower Industri)"
assert len(df_subcategories_split[df_subcategories_split["target_subcategory"] == "PDGR-ACP"]) == 4, "PDGR-ACP should have 4 items"
assert len(df_subcategories_split[df_subcategories_split["target_subcategory"] == "PDGR-ACN"]) == 8, "PDGR-ACN should have 8 items"
assert len(df_subcategories_split[df_subcategories_split["target_subcategory"] == "MES-GEN"]) == 1, "MES-GEN should have 1 item (Genset only)"
assert len(df_subcategories_split[df_subcategories_split["target_subcategory"] == "ELEK-MK"]) == 1, "ELEK-MK should have 1 item (Mesin Kopi only)"
assert len(df_subcategories_split[df_subcategories_split["target_subcategory"] == "ELEK-TG"]) == 11, "ELEK-TG should have 11 items (10 Topcon tools moved in)"
assert len(df_subcategories_split[df_subcategories_split["target_subcategory"] == "PRNT-PLT"]) == 2, "PRNT-PLT should have 2 items (Xerox A0 plotter moved in)"
assert len(df_subcategories_split[df_subcategories_split["target_subcategory"] == "ACC-STB"]) == 2, "ACC-STB should have 2 items (Scaffolding moved in)"

print("\n" + "="*50)
print("ALL INTEGRITY ASSERTIONS VERIFIED SUCCESSFULLY!")
print(f"  • Total Cleaned Assets: {len(df_assets_cleaned):,}")
print(f"  • Active Subcategory Assets Partitioned: {total_partition_rows:,}")
print(f"  • Permanently Deleted Misentries: {len(df_permanently_deleted):,}")
print(f"  • Subcategory Files Created: {len(manifest)}")
print(f"  • Duplicate IDs Across Files: 0")
print(f"  • Duplicate Codes Across Dataset: 0")
print("="*50)


Saved global cleaned assets to: tb_assets_list_cleaned.csv
Saved duplicate codes audit to: duplicated_codes_audit.csv
Saved 18 permanently deleted assets to: permanently_deleted_assets.csv
Saved partition manifest to: _split_manifest.json

ALL INTEGRITY ASSERTIONS VERIFIED SUCCESSFULLY!
  • Total Cleaned Assets: 6,962
  • Active Subcategory Assets Partitioned: 6,944
  • Permanently Deleted Misentries: 18
  • Subcategory Files Created: 111
  • Duplicate IDs Across Files: 0
  • Duplicate Codes Across Dataset: 0
